# ControlNet Poster Restyling — Stretch Phase (FR6)

**Run this notebook on Google Colab with a free T4 GPU** (Runtime → Change runtime type → GPU). It is deliberately self-contained: it does not import anything from the main `aiposter` app, and the main app never imports anything from here. A failure in this notebook — a Colab disconnect, an OOM, a bad generation — has zero effect on the live Streamlit demo. The only thing that crosses the boundary is a folder of finished PNG files you copy into the repo's `gallery/` directory afterward, read by the app's read-only Gallery tab.

**What this notebook does:**
1. Fetches a city's road network (via `osmnx`, the same library the main app uses) and exports it as a clean black-on-white lineart PNG — a ControlNet conditioning image.
2. Loads Stable Diffusion 1.5 + a ControlNet scribble/lineart model via `diffusers`, and restyles that street layout in 3 styles (watercolor, ink-wash, cyberpunk) across a few sample cities.
3. Saves the best results as a static image grid, plus individual PNGs in a `gallery/` output folder with the naming convention the app's Gallery tab expects: `{city}_{style}.png` (lowercase, spaces to underscores).

**Not run by Claude:** this notebook needs a real GPU runtime (Colab's T4), which isn't available in the environment that wrote this notebook. It's written carefully against the current `diffusers`/`transformers` APIs, but you'll be the first to actually execute it — if a cell errors, the fix is almost always a version pin in the install cell below.

In [ ]:
# 0. Install dependencies (Colab ships an old torch/diffusers by default)
!pip install -q "diffusers==0.31.0" "transformers==4.46.3" "accelerate==1.1.1" \
    "controlnet-aux==0.4.0" "safetensors==0.4.5" "osmnx==2.0.7" "opencv-python-headless"

In [ ]:
import os
import math
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import osmnx as ox
import torch
from PIL import Image

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- Runtime > Change runtime type > T4 GPU, then re-run.")

OUTPUT_ROOT = Path("/content/gallery_output")
LINEART_DIR = OUTPUT_ROOT / "lineart"
RESTYLED_DIR = OUTPUT_ROOT / "restyled"
GALLERY_DIR = OUTPUT_ROOT / "gallery"
for d in (LINEART_DIR, RESTYLED_DIR, GALLERY_DIR):
    d.mkdir(parents=True, exist_ok=True)

## Step 1 — Road-network lineart export

Fetches the road graph for a city via `osmnx.graph_from_point` (the same call the main app's `create_map_poster.py` makes) and plots it as pure black lines on a white background -- no color, no water/parks, no labels. This is the conditioning image ControlNet expects: a clean structural sketch of the layout it should follow.

In [ ]:
def fetch_lineart(city: str, country: str, dist: int = 3000, size_px: int = 512) -> Image.Image:
    """Black-on-white road-network lineart for one city, at a fixed square resolution
    (512x512 matches Stable Diffusion 1.5's native resolution, and what the ControlNet
    conditioning image should be sized to)."""
    point = ox.geocode(f"{city}, {country}")
    graph = ox.graph_from_point(point, dist=dist, dist_type="bbox", network_type="all", truncate_by_edge=True)
    graph_proj = ox.project_graph(graph)

    fig, ax = plt.subplots(figsize=(size_px / 100, size_px / 100), dpi=100)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")
    ax.set_position((0.0, 0.0, 1.0, 1.0))
    ox.plot_graph(
        graph_proj, ax=ax, bgcolor="white",
        node_size=0, edge_color="black", edge_linewidth=0.8,
        show=False, close=False,
    )
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")

    fig.canvas.draw()
    buf = fig.canvas.buffer_rgba()
    image = Image.frombuffer("RGBA", fig.canvas.get_width_height(), buf, "raw", "RGBA", 0, 1)
    plt.close(fig)
    return image.convert("RGB").resize((size_px, size_px))

In [ ]:
# A few sample cities -- swap or extend this list freely.
SAMPLE_CITIES = [
    ("Paris", "France"),
    ("Tokyo", "Japan"),
    ("Venice", "Italy"),
]

lineart_images = {}
for city, country in SAMPLE_CITIES:
    print(f"Fetching road network for {city}, {country}...")
    lineart = fetch_lineart(city, country)
    lineart_images[city] = lineart
    lineart.save(LINEART_DIR / f"{city.lower()}.png")
    display(lineart)

## Step 2 — ControlNet restyling

Loads `runwayml/stable-diffusion-v1-5` with `lllyasviel/sd-controlnet-scribble` (a scribble/lineart-conditioned ControlNet -- works well on the clean black-on-white road sketch from Step 1) and generates 3 stylistic variations of each city's layout.

In [ ]:
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-scribble", torch_dtype=torch.float16,
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", controlnet=controlnet, torch_dtype=torch.float16, safety_checker=None,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()  # keeps this comfortably inside a T4's 16GB

In [ ]:
STYLE_PROMPTS = {
    "watercolor": (
        "a delicate watercolor painting of a city street map, soft washes of color, "
        "artistic, gallery quality, high detail"
    ),
    "ink_wash": (
        "a traditional ink wash painting of a city street map, sumi-e style, monochrome "
        "black ink on rice paper, elegant brushstrokes"
    ),
    "cyberpunk": (
        "a neon cyberpunk city map at night, glowing pink and cyan streets, futuristic, "
        "high contrast, cinematic lighting"
    ),
}

NEGATIVE_PROMPT = "blurry, low quality, text, watermark, signature, distorted"

GENERATION_KWARGS = dict(
    num_inference_steps=25,
    guidance_scale=7.5,
    controlnet_conditioning_scale=1.0,
)

In [ ]:
restyled_images = {}
generator = torch.Generator(device="cuda").manual_seed(0)

for city, lineart in lineart_images.items():
    restyled_images[city] = {}
    for style, prompt in STYLE_PROMPTS.items():
        print(f"Generating {city} / {style}...")
        result = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            image=lineart,
            generator=generator,
            **GENERATION_KWARGS,
        )
        restyled = result.images[0]
        restyled_images[city][style] = restyled
        out_name = f"{city.lower()}_{style}.png"
        restyled.save(RESTYLED_DIR / out_name)
        display(restyled)

## Step 3 — Save the best results as a static image grid

Composes one grid image (rows = cities, columns = lineart + 3 styles) for a quick visual overview, and copies your chosen "best" result per city/style into `gallery/` using the exact filename convention the app's Gallery tab reads: `{city}_{style}.png`.

In [ ]:
def build_grid(lineart_images, restyled_images, styles, cell_size=256):
    cities = list(lineart_images)
    cols = 1 + len(styles)
    grid = Image.new("RGB", (cols * cell_size, len(cities) * cell_size), "white")
    for row, city in enumerate(cities):
        thumb = lineart_images[city].resize((cell_size, cell_size))
        grid.paste(thumb, (0, row * cell_size))
        for col, style in enumerate(styles, start=1):
            thumb = restyled_images[city][style].resize((cell_size, cell_size))
            grid.paste(thumb, (col * cell_size, row * cell_size))
    return grid

grid_image = build_grid(lineart_images, restyled_images, list(STYLE_PROMPTS))
grid_path = OUTPUT_ROOT / "gallery_grid.png"
grid_image.save(grid_path)
display(grid_image)
print(f"Saved grid to {grid_path}")

In [ ]:
# Copy every generated restyle into gallery/ using the app's expected naming convention.
# Review the grid above first and drop any city/style you don't want to ship -- everything
# left in restyled_images gets copied.
import shutil

for city, styles in restyled_images.items():
    for style, image in styles.items():
        out_name = f"{city.lower()}_{style}.png"
        image.save(GALLERY_DIR / out_name)

print(f"{sum(len(s) for s in restyled_images.values())} images ready in {GALLERY_DIR}")
print("Download this folder (or the individual files) and copy them into the repo's gallery/ directory.")

## Step 4 — Get the images into the app

Download the `gallery/` folder from the Colab file browser (left sidebar → folder icon → right-click `gallery_output/gallery` → Download), then copy its PNG files into this repo's `gallery/` directory at the root (next to `app.py`). The app's **Gallery** tab reads that directory directly -- no code changes needed, no GPU involved at runtime, and nothing here can break the live demo if this notebook's Colab session dies mid-run.